In [3]:
import numpy as np
import re
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document

# 1. Initialize local embedding engine
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 2. A document that changes topics mid-stream
raw_multitopic_doc = (
    "MeshQuery handles indexing on port 9092. "
    "All cluster nodes route internal tracking packets across this specific channel. "
    "Separately, our engineering team manages an aggressive US ETF investment portfolio. "
    "This financial corpus includes diversified allocations in indices like MOAT, IJK, and RSP. "
    "Finally, let's look at employee onboarding loops. "
    "New engineering hires must configure their local Ubuntu workspaces using our setup scripts."
)
print(type(raw_multitopic_doc))
print("✓ Cell 1: Multitopic technical document loaded.")

<class 'str'>
✓ Cell 1: Multitopic technical document loaded.


In [5]:
def calculate_cosine_distance(vec_a, vec_b):
    # Standard vector math to evaluate semantic divergence
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    similarity = dot_product / (norm_a * norm_b)
    return 1.0 - similarity  # Distance is the inverse of similarity

def generate_semantic_chunks(text: str, percentile_threshold: float = 85.0) -> list:
    # Step A: Split the block into raw sentences using basic regex boundary rules
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    if len(sentences) < 2:
        return [text]

    print(sentences)
        
    # Step B: Generate text vector embeddings for every sentence
    sentence_vectors = embeddings.embed_documents(sentences)
    
    # Step C: Compute the distance between consecutive sentences
    distances = []
    for i in range(len(sentence_vectors) - 1):
        dist = calculate_cosine_distance(sentence_vectors[i], sentence_vectors[i+1])
        distances.append(dist)
        
    # Step D: Determine the mathematical threshold boundary spike
    threshold = np.percentile(distances, percentile_threshold)
    
    print("\n--- Semantic Distance Divergence Report ---")
    chunks = []
    current_chunk = sentences[0]
    
    for i, dist in enumerate(distances):
        print(f"Gap between Sentence {i+1} and {i+2} | Distance: {dist:.4f} (Threshold: {threshold:.4f})")
        
        # If the difference in meaning handles a massive spike, break the group
        if dist >= threshold:
            print("🛑 [BREAK POINT IDENTIFIED - TOPIC SHIFT DETECTED]")
            chunks.append(current_chunk)
            current_chunk = sentences[i+1]
        else:
            current_chunk += " " + sentences[i+1]
            
    # Append final floating block
    chunks.append(current_chunk)
    return chunks

print("✓ Cell 2: Custom semantic chunking processor compiled successfully.")

✓ Cell 2: Custom semantic chunking processor compiled successfully.


In [7]:
# Compute semantic groups using an 85th percentile breakpoint trigger
final_semantic_chunks = generate_semantic_chunks(raw_multitopic_doc, percentile_threshold=85.0)

print("\n===========================================")
print("          FINAL SEMANTIC CHUNKS            ")
print("===========================================")
for idx, chunk in enumerate(final_semantic_chunks, 1):
    print(f"Chunk #{idx}:\n\"{chunk}\"\n-------------------------------------------")

['MeshQuery handles indexing on port 9092.', 'All cluster nodes route internal tracking packets across this specific channel.', 'Separately, our engineering team manages an aggressive US ETF investment portfolio.', 'This financial corpus includes diversified allocations in indices like MOAT, IJK, and RSP.', "Finally, let's look at employee onboarding loops.", 'New engineering hires must configure their local Ubuntu workspaces using our setup scripts.']

--- Semantic Distance Divergence Report ---
Gap between Sentence 1 and 2 | Distance: 0.5529 (Threshold: 0.5608)
Gap between Sentence 2 and 3 | Distance: 0.5728 (Threshold: 0.5608)
🛑 [BREAK POINT IDENTIFIED - TOPIC SHIFT DETECTED]
Gap between Sentence 3 and 4 | Distance: 0.4631 (Threshold: 0.5608)
Gap between Sentence 4 and 5 | Distance: 0.5455 (Threshold: 0.5608)
Gap between Sentence 5 and 6 | Distance: 0.4223 (Threshold: 0.5608)

          FINAL SEMANTIC CHUNKS            
Chunk #1:
"MeshQuery handles indexing on port 9092. All cluster